# 08. Foundation Models for Time Series: Zero-Shot Forecasting

## Background

The success of large language models at zero-shot transfer has inspired analogous approaches for time series. If a model trained on a large, diverse corpus of time series can learn universal temporal patterns, it could forecast new series without any fine-tuning — analogous to GPT-4 answering questions about topics not in its fine-tuning data.

**Chronos** (Amazon, 2024): treats time series forecasting as language modeling. Quantize time series values into bins (like tokens), then train a T5-style encoder-decoder transformer to predict the next tokens. Zero-shot on new series.

**TimesFM** (Google, 2024): 200M parameter model trained on 100B time points from real and synthetic data. Uses patch tokenization (like PatchTST) but at scale.

**Moirai** (Salesforce, 2024): Universal time series model supporting multiple frequencies (hourly, daily, weekly, monthly) via frequency embeddings.

## What You'll Learn

- How Chronos tokenizes continuous values as discrete tokens
- Zero-shot forecasting pipeline with Chronos
- Comparing zero-shot vs fine-tuned models on a benchmark
- When foundation models beat supervised models (and when they don't)

## Why This Matters

Foundation models change the cost-benefit of forecasting: instead of collecting historical data, engineering features, and training per-series models, you can deploy a zero-shot foundation model and iterate. The trade-off: foundation models excel on 'typical' time series patterns but may lag fine-tuned models on domain-specific series with unusual dynamics.


## Chronos-Style Value Tokenization

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import pandas as pd

class ChronosTokenizer:
    '''Chronos-style value quantization: map continuous values to discrete tokens.

    Strategy:
    1. Normalize series to zero mean, unit std
    2. Clip to [-15, 15] (handles outliers)
    3. Quantize to n_bins uniform bins
    4. Special tokens: PAD, BOS (beginning of sequence)
    '''

    PAD_TOKEN = 0
    BOS_TOKEN = 1

    def __init__(self, n_bins: int = 4096, clip_range: float = 15.0):
        self.n_bins = n_bins
        self.clip_range = clip_range
        self.n_special = 2  # PAD + BOS
        self.vocab_size = n_bins + self.n_special

        # Bin boundaries: uniform in [-clip, clip]
        self.boundaries = np.linspace(-clip_range, clip_range, n_bins - 1)
        self.centers = np.concatenate([[-clip_range],
                                        (self.boundaries[:-1] + self.boundaries[1:]) / 2,
                                        [clip_range]])

    def encode(self, series: np.ndarray) -> np.ndarray:
        '''Normalize and quantize to token IDs.'''
        # Instance normalization
        mean = np.mean(series)
        std = np.std(series) + 1e-8
        normalized = (series - mean) / std

        # Clip and digitize
        clipped = np.clip(normalized, -self.clip_range, self.clip_range)
        bin_ids = np.digitize(clipped, self.boundaries)  # 0 to n_bins-1

        # Shift by n_special to leave room for special tokens
        token_ids = bin_ids + self.n_special
        return token_ids, mean, std

    def decode(self, token_ids: np.ndarray, mean: float, std: float) -> np.ndarray:
        '''Convert token IDs back to continuous values.'''
        bin_ids = token_ids - self.n_special
        bin_ids = np.clip(bin_ids, 0, self.n_bins - 1)
        normalized_values = self.centers[bin_ids]
        return normalized_values * std + mean

tokenizer = ChronosTokenizer(n_bins=4096)

# Example series
series = np.array([100, 103, 107, 115, 122, 118, 112, 108, 105, 101, 99, 97])
token_ids, mean, std = tokenizer.encode(series)
reconstructed = tokenizer.decode(token_ids, mean, std)

print(f'Chronos Tokenization:')
print(f'  Vocab size: {tokenizer.vocab_size} (4096 bins + 2 special)')
print(f'  Series:       {series}')
print(f'  Token IDs:    {token_ids}')
print(f'  Reconstructed:{np.round(reconstructed, 1)}')
print(f'  Max error:    {np.max(np.abs(series - reconstructed)):.4f}')
print(f'\nQuantization accuracy:')
print(f'  Each token covers ~{2*tokenizer.clip_range/tokenizer.n_bins:.4f} std units')


## Zero-Shot Forecasting Pipeline

In [ ]:
class MinimalChronosModel(nn.Module):
    '''Minimal Chronos-style model for demonstration.
    Real Chronos: T5 encoder-decoder, 46M-700M params.
    This: smaller transformer, same design pattern.
    '''

    def __init__(self, vocab_size: int, d_model: int = 64,
                 n_heads: int = 4, n_layers: int = 3):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Parameter(torch.randn(1, 512, d_model) * 0.02)

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model*4,
            batch_first=True, norm_first=True
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=n_layers)

        # Output: predict next token logits
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, context_tokens: torch.Tensor) -> torch.Tensor:
        T = context_tokens.shape[1]
        emb = self.token_embed(context_tokens) + self.pos_embed[:, :T, :]

        # Causal mask (autoregressive)
        causal_mask = nn.Transformer.generate_square_subsequent_mask(T, device=emb.device)
        out = self.decoder(emb, emb, tgt_mask=causal_mask, memory_mask=causal_mask)
        return self.lm_head(out)  # (B, T, vocab_size)

    def forecast(self, context_tokens: torch.Tensor, horizon: int,
                  n_samples: int = 20, temperature: float = 1.0) -> torch.Tensor:
        '''Autoregressive sampling for probabilistic forecasts.'''
        B, T = context_tokens.shape
        all_samples = []

        for _ in range(n_samples):
            tokens = context_tokens.clone()
            for h in range(horizon):
                logits = self(tokens)[:, -1, :] / temperature  # (B, vocab)
                next_token = torch.multinomial(torch.softmax(logits, dim=-1), 1)
                tokens = torch.cat([tokens, next_token], dim=1)
            all_samples.append(tokens[:, -horizon:])

        return torch.stack(all_samples, dim=1)  # (B, n_samples, horizon)

tokenizer = ChronosTokenizer(n_bins=512)  # smaller for demo
model = MinimalChronosModel(vocab_size=tokenizer.vocab_size, d_model=64)
params = sum(p.numel() for p in model.parameters())
print(f'Minimal Chronos model: {params:,} parameters')

# Generate synthetic series and run zero-shot forecast
np.random.seed(42)
test_series = 100 + 0.1*np.arange(100) + 10*np.sin(2*np.pi*np.arange(100)/52) + np.random.randn(100)*2

context = test_series[:80]
actuals = test_series[80:]

# Tokenize and forecast
token_ids, mean, std = tokenizer.encode(context)
context_tensor = torch.tensor(token_ids, dtype=torch.long).unsqueeze(0)

with torch.no_grad():
    sample_tokens = model.forecast(context_tensor, horizon=20, n_samples=100)

# Decode samples to values
sample_values = np.array([
    tokenizer.decode(sample_tokens[0, s].numpy(), mean, std)
    for s in range(100)
])

q10 = np.percentile(sample_values, 10, axis=0)
q50 = np.percentile(sample_values, 50, axis=0)
q90 = np.percentile(sample_values, 90, axis=0)

coverage = np.mean((actuals >= q10) & (actuals <= q90))
mape = np.mean(np.abs((actuals - q50) / actuals)) * 100

print(f'\nZero-shot forecast (untrained model — baseline only):')
print(f'  Horizon: {len(actuals)} steps')
print(f'  Median MAPE: {mape:.2f}%')
print(f'  80% PI coverage: {coverage:.1%} (target 80%)')
print(f'\nNote: Real Chronos is trained on 100B time points from diverse sources')
print(f'and achieves competitive zero-shot accuracy out of the box.')
